In [ ]:
import re
from nltk.corpus import stopwords
import string
import pandas as pd

from bs4 import BeautifulSoup # For removing HTML

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='bs4')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dataset

In [ ]:
#Load data into df
df = pd.read_csv("/content/drive/My Drive/NLP_DL/NLP/data/sample.csv")
df.head(10)

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,2,airplane%20accident,Mumbai india,Horrible Accident | Man Died In Wings of Airp...,1
2,3,ambulance,Amsterdam,http://t.co/7xGLah10zL Twelve feared killed in...,1
3,4,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0
4,5,NaN,NaN,<b>All residents</b> asked to 'shelter in plac...,1
5,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
6,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
7,8,NaN,NaN,emergency@gmail.com #RockyFire Update => Cali...,1
8,10,NaN,NaN,#flood #disaster Heavy rain causes flash flood...,1
9,13,NaN,NaN,I'm on top of the hill and I can see a fire in...,1


In [ ]:
df.shape

(999, 5)

If we check the description of the competition, we can observe that the keywords are important for the classification of distaster tweet and hence a combined tweet column is created by joining keyword and text. First the empty keywords are replaced by "".

#**Combine keyword with tweet text**

In [ ]:
# Replace missing values in the 'keyword' column with an empty string
df["keyword"] = df["keyword"].fillna("")

# Combine the keyword and tweet text into a single text field
df["tweet"] = df["keyword"] + " " + df["text"]

# Display first 10 tweets
df.head(10)

,id,keyword,location,text,target,tweet
0,1,,NaN,Our Deeds are the Reason of this #earthquake M...,1,Our Deeds are the Reason of this #earthquake ...
1,2,airplane%20accident,Mumbai india,Horrible Accident | Man Died In Wings of Airp...,1,airplane%20accident Horrible Accident | Man D...
2,3,ambulance,Amsterdam,http://t.co/7xGLah10zL Twelve feared killed in...,1,ambulance http://t.co/7xGLah10zL Twelve feared...
3,4,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0,ablaze @PhDSquares #mufc they've built so much...
4,5,,NaN,<b>All residents</b> asked to 'shelter in plac...,1,<b>All residents</b> asked to 'shelter in pla...
5,6,,NaN,"13,000 people receive #wildfires evacuation or...",1,"13,000 people receive #wildfires evacuation o..."
6,7,,NaN,Just got sent this photo from Ruby #Alaska as ...,1,Just got sent this photo from Ruby #Alaska as...
7,8,,NaN,emergency@gmail.com #RockyFire Update => Cali...,1,emergency@gmail.com #RockyFire Update => Cal...
8,10,,NaN,#flood #disaster Heavy rain causes flash flood...,1,#flood #disaster Heavy rain causes flash floo...
9,13,,NaN,I'm on top of the hill and I can see a fire in...,1,I'm on top of the hill and I can see a fire i...


In [ ]:
# Convert all characters in the 'tweet' column to lowercase
# and store the result in a new column called 'tweet_lower'
df["tweet_lower"] = df["tweet"].str.lower()

# Display first 10 tweets
df["tweet_lower"].head(10)

,tweet_lower
0,our deeds are the reason of this #earthquake ...
1,airplane%20accident horrible accident | man d...
2,ambulance http://t.co/7xglah10zl twelve feared...
3,ablaze @phdsquares #mufc they've built so much...
4,<b>all residents</b> asked to 'shelter in pla...
5,"13,000 people receive #wildfires evacuation o..."
6,just got sent this photo from ruby #alaska as...
7,emergency@gmail.com #rockyfire update => cal...
8,#flood #disaster heavy rain causes flash floo...
9,i'm on top of the hill and i can see a fire i...


In [ ]:
df.head(10)

,id,keyword,location,text,target,tweet,tweet_lower
0,1,,NaN,Our Deeds are the Reason of this #earthquake M...,1,Our Deeds are the Reason of this #earthquake ...,our deeds are the reason of this #earthquake ...
1,2,airplane%20accident,Mumbai india,Horrible Accident | Man Died In Wings of Airp...,1,airplane%20accident Horrible Accident | Man D...,airplane%20accident horrible accident | man d...
2,3,ambulance,Amsterdam,http://t.co/7xGLah10zL Twelve feared killed in...,1,ambulance http://t.co/7xGLah10zL Twelve feared...,ambulance http://t.co/7xglah10zl twelve feared...
3,4,ablaze,Pretoria,@PhDSquares #mufc they've built so much hype a...,0,ablaze @PhDSquares #mufc they've built so much...,ablaze @phdsquares #mufc they've built so much...
4,5,,NaN,<b>All residents</b> asked to 'shelter in plac...,1,<b>All residents</b> asked to 'shelter in pla...,<b>all residents</b> asked to 'shelter in pla...
5,6,,NaN,"13,000 people receive #wildfires evacuation or...",1,"13,000 people receive #wildfires evacuation o...","13,000 people receive #wildfires evacuation o..."
6,7,,NaN,Just got sent this photo from Ruby #Alaska as ...,1,Just got sent this photo from Ruby #Alaska as...,just got sent this photo from ruby #alaska as...
7,8,,NaN,emergency@gmail.com #RockyFire Update => Cali...,1,emergency@gmail.com #RockyFire Update => Cal...,emergency@gmail.com #rockyfire update => cal...
8,10,,NaN,#flood #disaster Heavy rain causes flash flood...,1,#flood #disaster Heavy rain causes flash floo...,#flood #disaster heavy rain causes flash floo...
9,13,,NaN,I'm on top of the hill and I can see a fire in...,1,I'm on top of the hill and I can see a fire i...,i'm on top of the hill and i can see a fire i...


#**Remove HTML**

In [ ]:
def remove_html(text):
    soup = BeautifulSoup(text)
    text = soup.get_text()
    return text

In [ ]:
# Remove HTML tags from each tweet using the 'remove_html' function
# and store the cleaned text in a new column
df["tweet_noHTML"] = df["tweet_lower"].apply(remove_html)

# Display the first 10 tweets after removing HTML tags
df["tweet_noHTML"].head(10)

,tweet_noHTML
0,our deeds are the reason of this #earthquake m...
1,airplane%20accident horrible accident | man d...
2,ambulance http://t.co/7xglah10zl twelve feared...
3,ablaze @phdsquares #mufc they've built so much...
4,all residents asked to 'shelter in place' are ...
5,"13,000 people receive #wildfires evacuation or..."
6,just got sent this photo from ruby #alaska as ...
7,emergency@gmail.com #rockyfire update => calif...
8,#flood #disaster heavy rain causes flash flood...
9,i'm on top of the hill and i can see a fire in...


#**Expand Contractions**

In [ ]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00


In [ ]:
# Import the contractions library to expand shortened forms
# such as "can't" → "cannot" and "I'm" → "I am"
import contractions

# Expand contractions in each tweet and store the cleaned text
# in a new column called 'tweet_noContractions'
df["tweet_noContractions"] = df["tweet_noHTML"].apply(contractions.fix)

# Display the first 10 tweets after expanding contractions
df["tweet_noContractions"].head(10)

,tweet_noContractions
0,our deeds are the reason of this #earthquake m...
1,airplane%20accident horrible accident | man d...
2,ambulance http://t.co/7xglah10zl twelve feared...
3,ablaze @phdsquares #mufc they have built so mu...
4,all residents asked to 'shelter in place' are ...
5,"13,000 people receive #wildfires evacuation or..."
6,just got sent this photo from ruby #alaska as ...
7,emergency@gmail.com #rockyfire update => calif...
8,#flood #disaster heavy rain causes flash flood...
9,i am on top of the hill and i can see a fire i...


#**Remove URLs**

In [ ]:
def remove_urls(text):
    # Define a pattern to identify URLs starting with http:// or https://
    pattern = re.compile(r'https?://\S+')

    # Replace all URLs found in the text with an empty string
    text = re.sub(pattern, "", text)

    # Return the text after removing URLs
    return text

In [ ]:
df["tweet_noURLs"] = df["tweet_noContractions"].apply(remove_urls)
df["tweet_noURLs"].head(10)

,tweet_noURLs
0,our deeds are the reason of this #earthquake m...
1,airplane%20accident horrible accident | man d...
2,ambulance twelve feared killed in pakistani a...
3,ablaze @phdsquares #mufc they have built so mu...
4,all residents asked to 'shelter in place' are ...
5,"13,000 people receive #wildfires evacuation or..."
6,just got sent this photo from ruby #alaska as ...
7,emergency@gmail.com #rockyfire update => calif...
8,#flood #disaster heavy rain causes flash flood...
9,i am on top of the hill and i can see a fire i...


#**Remove Email IDs**

In [ ]:
def remove_emails(text):
    # Define a pattern to identify email addresses
    pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")

    # Remove all email addresses from the text
    text = re.sub(pattern, "", text)

    # Return the cleaned text
    return text

In [ ]:
df["tweet_noEmail"] = df["tweet_noURLs"].apply(remove_emails)
df["tweet_noEmail"].head(10)

,tweet_noEmail
0,our deeds are the reason of this #earthquake m...
1,airplane%20accident horrible accident | man d...
2,ambulance twelve feared killed in pakistani a...
3,ablaze @phdsquares #mufc they have built so mu...
4,all residents asked to 'shelter in place' are ...
5,"13,000 people receive #wildfires evacuation or..."
6,just got sent this photo from ruby #alaska as ...
7,#rockyfire update => california hwy. 20 close...
8,#flood #disaster heavy rain causes flash flood...
9,i am on top of the hill and i can see a fire i...


In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

df[["tweet_noEmail"]].head(10)

,tweet_noEmail
0,our deeds are the reason of this #earthquake may allah forgive us all
1,airplane%20accident horrible accident | man died in wings of airplaneåê(29-07-2015)
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze @phdsquares #mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season.
4,all residents asked to 'shelter in place' are being notified by officers. no other evacuation or shelter in place orders are expected
5,"13,000 people receive #wildfires evacuation orders in california"
6,just got sent this photo from ruby #alaska as smoke from #wildfires pours into a school
7,#rockyfire update => california hwy. 20 closed in both directions due to lake county fire - #cafire #wildfires
8,"#flood #disaster heavy rain causes flash flooding of streets in manitou, colorado springs areas"
9,i am on top of the hill and i can see a fire in the woods...


#**Twitter mentions (e.g., @username)**

In [ ]:
def remove_mentions(text):
    pattern = re.compile(r"@\w+")
    text = re.sub(pattern, "", text)
    return text

In [ ]:
df["tweet_noMention"] = df["tweet_noEmail"].apply(remove_mentions)
df["tweet_noMention"].head(10)

,tweet_noMention
0,our deeds are the reason of this #earthquake may allah forgive us all
1,airplane%20accident horrible accident | man died in wings of airplaneåê(29-07-2015)
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze #mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season.
4,all residents asked to 'shelter in place' are being notified by officers. no other evacuation or shelter in place orders are expected
5,"13,000 people receive #wildfires evacuation orders in california"
6,just got sent this photo from ruby #alaska as smoke from #wildfires pours into a school
7,#rockyfire update => california hwy. 20 closed in both directions due to lake county fire - #cafire #wildfires
8,"#flood #disaster heavy rain causes flash flooding of streets in manitou, colorado springs areas"
9,i am on top of the hill and i can see a fire in the woods...


<h2><strong>Retain Hashtags</strong></h2>

<p>
  Although <strong>hashtags</strong> can be removed like other special tokens,
  they are <strong>retained</strong> in this task because they contain
  <strong>valuable contextual information</strong>. Hashtags often describe the
  <strong>topic</strong> or <strong>type of disaster</strong>, making them useful
  features for improving <strong>classification performance</strong>.
</p>

<h2><strong>Handling Emojis</strong></h2>

<p>Generally, <strong>emojis are removed</strong> during preprocessing. However, in <strong>disaster-related tweets</strong>, they may provide useful emotional information.</p>

<p>One approach is to map emojis to <strong>basic emotions</strong> such as happiness, sadness, anger, disgust, fear, surprise, and neutral. Another option is to replace each emoji with its <strong>text description</strong> (e.g., 😀 → “smiling face”).</p>

<p>This step should be performed <strong>before removing Unicode characters</strong>, since emojis are represented in Unicode.</p>




In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 26.4 MB/s eta 0:00:00


In [ ]:
import emoji

def replace_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

In [ ]:
text = "The situation is terrible 😢 Please help! 🙏"
print(replace_emojis(text))

The situation is terrible  crying_face  Please help!  folded_hands 


<h2><strong>Handling Accented Words</strong></h2>

<p><strong>Accented characters</strong> are common in borrowed words and proper names, such as <em>résumé</em> and <em>tête-à-tête</em>, and are frequently used in languages such as Spanish, French, Italian, German, and Portuguese.</p>

<p>These characters should be handled <strong>before removing Unicode characters</strong>; otherwise, they may be lost. However, for <strong>disaster tweet classification</strong>, converting accented characters to ASCII may not always be useful because the text is often <strong>noisy</strong> and may result in <strong>meaningless text</strong>.</p>


In [ ]:
!pip install unidecode
from unidecode import unidecode
text = "words of foreign origin, such as résumé and tête-à-tête"
unidecode(text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 13.6 MB/s eta 0:00:00


'words of foreign origin, such as resume and tete-a-tete'

In [ ]:
def handle_accents(text):
    text = unidecode(text)
    return text

In [ ]:
df["tweet_handleAccents"] = df["tweet_noMention"].apply(handle_accents)
df["tweet_handleAccents"].head(10)

,tweet_handleAccents
0,our deeds are the reason of this #earthquake may allah forgive us all
1,airplane%20accident horrible accident | man died in wings of airplaneae(29-07-2015)
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze #mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season.
4,all residents asked to 'shelter in place' are being notified by officers. no other evacuation or shelter in place orders are expected
5,"13,000 people receive #wildfires evacuation orders in california"
6,just got sent this photo from ruby #alaska as smoke from #wildfires pours into a school
7,#rockyfire update => california hwy. 20 closed in both directions due to lake county fire - #cafire #wildfires
8,"#flood #disaster heavy rain causes flash flooding of streets in manitou, colorado springs areas"
9,i am on top of the hill and i can see a fire in the woods...


#**Remove Unicode Characters**

In [ ]:
def remove_unicode_chars(text):
    text = text.encode("ascii", "ignore").decode()
    return text

As mentioned before, the accented characters are removed by this step. For example.

In [ ]:
text = "words of foreign origin, such as résumé and tête-à-tête"
remove_unicode_chars(text)

'words of foreign origin, such as rsum and tte--tte'

In [ ]:
df["tweet_noUnicode"] = df["tweet_noMention"].apply(remove_unicode_chars)
df["tweet_noUnicode"].head(10)

,tweet_noUnicode
0,our deeds are the reason of this #earthquake may allah forgive us all
1,airplane%20accident horrible accident | man died in wings of airplane(29-07-2015)
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze #mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season.
4,all residents asked to 'shelter in place' are being notified by officers. no other evacuation or shelter in place orders are expected
5,"13,000 people receive #wildfires evacuation orders in california"
6,just got sent this photo from ruby #alaska as smoke from #wildfires pours into a school
7,#rockyfire update => california hwy. 20 closed in both directions due to lake county fire - #cafire #wildfires
8,"#flood #disaster heavy rain causes flash flooding of streets in manitou, colorado springs areas"
9,i am on top of the hill and i can see a fire in the woods...


<h2><strong>Abbreviation/Acronym Disambiguation</strong></h2>

<p>Disaster tweets contain many <strong>abbreviations and acronyms</strong> that may carry important information for classification. These terms can be <strong>distorted or removed</strong> during preprocessing, so they should be expanded early in the process.</p>

<p>Several common abbreviations are documented in <strong>@gunesevitan’s notebook</strong>. I am also exploring an approach to identify and disambiguate abbreviations in the <strong>Disaster Tweets dataset</strong>.</p>


In [ ]:
# Acronyms
def remove_abbreviations(text):
    text = re.sub(r"mh370", "missing malaysia airlines flight", text)
    text = re.sub(r"okwx", "oklahoma city weather", text)
    text = re.sub(r"arwx", "arkansas weather", text)
    text = re.sub(r"gawx", "georgia weather", text)
    text = re.sub(r"scwx", "south carolina weather", text)
    text = re.sub(r"cawx", "california weather", text)
    text = re.sub(r"tnwx", "tennessee weather", text)
    text = re.sub(r"azwx", "arizona weather", text)
    text = re.sub(r"alwx", "alabama Weather", text)
    text = re.sub(r"wordpressdotcom", "wordpress", text)
    text = re.sub(r"usnwsgov", "united states national weather service", text)
    text = re.sub(r"suruc", "sanliurfa", tweet)
    return text

<p>There are <strong>many more abbreviations</strong> in the dataset, so a thorough review is required to identify them all.</p>

<p>During normalization, different forms of the same abbreviation should be <strong>mapped to a common form</strong>. For example, <strong>U.S.A.</strong> and <strong>USA</strong> refer to the same entity and can be standardized as <strong>USA</strong>.</p>

<p>This step should be performed <strong>before lowercasing</strong> to take advantage of capitalization information.</p>


In [ ]:
def normalize_abbreviations(text):
    matches = re.finditer(r"([A-Z]\.)+", text)
    matched_abbr = [match.group() for match in matches]
    for abbr in matched_abbr:
        text = re.sub(abbr,abbr.replace(".",""), text)
    return text

In [ ]:
text = "I.S.R.O. is Indian aerospace agency similar to N.A.S.A. in U.S.A."
normalize_abbreviations(text)

'ISRO is Indian aerospace agency similar to NASA in USA'

<h2><strong>Handle Punctuations</strong></h2>

<p><strong>Punctuation marks</strong> help define text structure and can be useful for <strong>sentence tokenization</strong>. However, for some NLP tasks, they may not provide useful information and can be removed.</p>

<p>In this preprocessing step, <strong>Python’s built-in regular expression library</strong> is used to remove punctuation marks.</p>


In [ ]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
def remove_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), " ",text)
    return text

In [ ]:
string.punctuation.replace("&","")

'!"#$%\'()*+,-./:;<=>?@[\\]^_`{|}~'

Another approach might be to only keep alphanumeric characters using regex pattern "[^a-zA-Z0-9]".

In [ ]:
df["tweet_noPuncts"] = df["tweet_noUnicode"].apply(remove_punctuations)
df["tweet_noPuncts"].head(10)

,tweet_noPuncts
0,our deeds are the reason of this earthquake may allah forgive us all
1,airplane 20accident horrible accident man died in wings of airplane 29 07 2015
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season
4,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in place orders are expected
5,13 000 people receive wildfires evacuation orders in california
6,just got sent this photo from ruby alaska as smoke from wildfires pours into a school
7,rockyfire update california hwy 20 closed in both directions due to lake county fire cafire wildfires
8,flood disaster heavy rain causes flash flooding of streets in manitou colorado springs areas
9,i am on top of the hill and i can see a fire in the woods


Sometimes, the punctuations along with digits represent certain important information such as currency ($12.34) or percentages (75\%). Such occurances can be replaced with words like "money amount" and "percentage".

In [ ]:
def handle_amount_and_percentage(text):
    text = re.sub(r"(₹|\$|£|€|¥)\s?\d+(\.\d+)?", "money amount",text)
    text = re.sub(r"\d+(\.\d+)?\s?%", "percentage",text)
    return text

In [ ]:
text = "₹100 is $ 1.22 which is which is 6.15% less than last year"
handle_amount_and_percentage(text)

'money amount is money amount which is which is percentage less than last year'

This can be modified further for replacing individual currencies with their names such as rupee, dollar, pound, euro, yen, etc.

<h2><strong>Handling Digits or Words Containing Digits</strong></h2>

<p>Removing <strong>digits or words containing digits</strong> may not always be appropriate. For example, <strong>MH370</strong> refers to Malaysia Airlines Flight 370, which is highly relevant in disaster-related tweets.</p>

<p>Therefore, <strong>meaningful alphanumeric terms</strong> should be retained when they provide useful information for classification.</p>


In [ ]:
def remove_digits(text):
    pattern = re.compile("\\w*\\d+\\w*")
    text = re.sub(pattern, "",text)
    return text

In [ ]:
text = " m194 0104 utc5km s of volcano hawaii"
remove_digits(text)

'    s of volcano hawaii'

In [ ]:
df["tweet_noDigits"] = df["tweet_noPuncts"].apply(remove_digits)
df["tweet_noDigits"].head(10)

,tweet_noDigits
0,our deeds are the reason of this earthquake may allah forgive us all
1,airplane horrible accident man died in wings of airplane
2,ambulance twelve feared killed in pakistani air ambulance helicopter crash
3,ablaze mufc they have built so much hype around new acquisitions but i doubt they will set the epl ablaze this season
4,all residents asked to shelter in place are being notified by officers no other evacuation or shelter in place orders are expected
5,people receive wildfires evacuation orders in california
6,just got sent this photo from ruby alaska as smoke from wildfires pours into a school
7,rockyfire update california hwy closed in both directions due to lake county fire cafire wildfires
8,flood disaster heavy rain causes flash flooding of streets in manitou colorado springs areas
9,i am on top of the hill and i can see a fire in the woods


<h2><strong>Remove Stopwords</strong></h2>

<p><strong>Stopword removal</strong> is a common NLP preprocessing step. Stopwords are frequently used words such as <strong>“a”, “and”, “the”, “is”, and “can”</strong> that are often removed to retain more <strong>information-rich words</strong>.</p>


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
print(stop_words)

{"hasn't", 'before', "he'd", 'each', "she's", "they've", 'very', 's', 'from', 'we', 'on', 'ain', 'she', 'few', 'over', "i'd", 'same', "i've", 'they', 'his', 'under', "they're", 'can', 'because', 'll', 'while', 'i', 'any', "don't", 'himself', "you've", 'itself', 'why', "shan't", 'doesn', 'off', 'he', "should've", "they'd", 'so', 'then', 'it', 'haven', "mightn't", 'them', 'this', 'whom', 'which', 'how', 'those', "it'd", 'shan', 'does', "it's", 'too', 'do', 'wouldn', 'into', "shouldn't", 'my', 'am', 'were', 'has', 'what', 'other', 'further', "needn't", 'having', "you're", 'during', 'there', 'against', "doesn't", 'themselves', 'a', 're', 'an', 'didn', 'mightn', 'did', 'is', 'him', 'its', 'or', 'but', "aren't", "mustn't", "won't", 'shouldn', "wouldn't", 'more', 'as', 'couldn', 'should', 'out', "hadn't", "she'd", "weren't", 'don', "that'll", 'again', 'needn', 'ourselves', "she'll", 'that', 'above', 'ours', 'hadn', 'all', 'have', "he's", 'me', 'weren', 'most', 'now', 'who', 'own', 'their', 'v

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

In [ ]:
df["tweet_noStopwords"] = df["tweet_noDigits"].apply(remove_stopwords)
df["tweet_noStopwords"].head(10)

,tweet_noStopwords
0,deeds reason earthquake may allah forgive us
1,airplane horrible accident man died wings airplane
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze mufc built much hype around new acquisitions doubt set epl ablaze season
4,residents asked shelter place notified officers evacuation shelter place orders expected
5,people receive wildfires evacuation orders california
6,got sent photo ruby alaska smoke wildfires pours school
7,rockyfire update california hwy closed directions due lake county fire cafire wildfires
8,flood disaster heavy rain causes flash flooding streets manitou colorado springs areas
9,top hill see fire woods


#**Removing Extra Spaces**



In [ ]:
def remove_extra_spaces(text):
    text = re.sub(' +', ' ', text).strip()
    return text

In [ ]:
df["tweet_noExtraspace"] = df["tweet_noStopwords"].apply(remove_extra_spaces)
df["tweet_noExtraspace"].head(10)

,tweet_noExtraspace
0,deeds reason earthquake may allah forgive us
1,airplane horrible accident man died wings airplane
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze mufc built much hype around new acquisitions doubt set epl ablaze season
4,residents asked shelter place notified officers evacuation shelter place orders expected
5,people receive wildfires evacuation orders california
6,got sent photo ruby alaska smoke wildfires pours school
7,rockyfire update california hwy closed directions due lake county fire cafire wildfires
8,flood disaster heavy rain causes flash flooding streets manitou colorado springs areas
9,top hill see fire woods


#**Stemming or Lemmatization**


In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = [lemmatizer.lemmatize(word) for word in text.split()]
    text = ' '.join(words)
    return text

In [ ]:
import nltk
nltk.download('wordnet')
df["tweet_lemmatised"] = df["tweet_noExtraspace"].apply(lemmatize_text)
df["tweet_lemmatised"].head(10)

[nltk_data] Downloading package wordnet to /root/nltk_data...


,tweet_lemmatised
0,deed reason earthquake may allah forgive u
1,airplane horrible accident man died wing airplane
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze mufc built much hype around new acquisition doubt set epl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rockyfire update california hwy closed direction due lake county fire cafire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


<h2><strong>Spelling Correction</strong></h2>

<p>Tools such as <strong>TextBlob</strong> and <strong>SymSpellPy</strong> can be used for spelling correction. However, TextBlob is relatively <strong>slow</strong>, while SymSpellPy is <strong>fast and accurate</strong>. It is also <strong>language-agnostic</strong> when an appropriate dictionary is provided, making it a suitable choice here.</p>


In [ ]:
!pip install symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 31.1 MB/s eta 0:00:00


In [ ]:
import pkg_resources
from symspellpy import SymSpell, Verbosity

/tmp/ipykernel_2554/1266291330.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


SymSpellpy give multiple suggestions to the words for spelling correction. We can select the first suggested word having highest probability.

In [ ]:
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

True

In [ ]:
def correct_spelling_symspell(text):
    words = [
        sym_spell.lookup(
            word,
            Verbosity.CLOSEST,
            max_edit_distance=2,
            include_unknown=True
            )[0].term
        for word in text.split()]
    text = " ".join(words)
    return text

The `include_unknown` option keeps the words not within `max_edit_distance` from the words in the dictionary

In [ ]:
df["tweet_spellcheck"] = df["tweet_lemmatised"].apply(correct_spelling_symspell)
df["tweet_spellcheck"].head(10)

,tweet_spellcheck
0,deed reason earthquake may allah forgive a
1,airline horrible accident man died wing airline
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze muff built much hype around new acquisition doubt set cpl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rockyfire update california hwy closed direction due lake county fire afire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


It can be observed that it is not perfect and introduces more stopwords but can help in many cases. Some more investigation is required with the competition solution results

The [symspellpy library](https://symspellpy.readthedocs.io/en/latest/examples/dictionary.html) is said be "language independent (agnostic)" and can be used with any language. The already available english dictionary is used in the above example, but such a dictionary can be easily created for any language using large enough text data in 'plain text' format using the `create_dictionary` function. You can read [1000x Faster Spelling Correction algorithm](https://wolfgarbe.medium.com/1000x-faster-spelling-correction-algorithm-2012-8701fcd87a5f) and the [documentation of symspellpy library](https://symspellpy.readthedocs.io/en/latest/) for more details.

In [ ]:
# from symspellpy import SymSpell

# sym_spell = SymSpell()
# corpus_path = <path/to/plain/text/file>
# sym_spell.create_dictionary(corpus_path)

# print(sym_spell.words)

#**Correcting Compounded Words**


In [ ]:
bigram_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_bigramdictionary_en_243_342.txt"
)
sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

True

In [ ]:
def correct_spelling_symspell_compound(text):
    words = [
        sym_spell.lookup_compound(
            word,
            max_edit_distance=2
            )[0].term
        for word in text.split()]
    text = " ".join(words)
    return text

In [ ]:
text = "IranDeal PantherAttack TrapMusic StrategicPatience socialnews NASAHurricane onlinecommunities humanconsumption"
correct_spelling_symspell_compound(text)

'iran deal panther attack trap music strategic patience social news as hurricane online communities human consumption'

In [ ]:
df["tweet_spellcheck_compound"] = df["tweet_spellcheck"].apply(correct_spelling_symspell_compound)
df["tweet_spellcheck_compound"].head(10)

,tweet_spellcheck_compound
0,deed reason earthquake may allah forgive a
1,airline horrible accident man died wing airline
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze muff built much hype around new acquisition doubt set cpl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rock fire update california hwy closed direction due lake county fire afire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


## Britsh-American English Conversion
There are many words in British and American english which differ in spellings such as (colour: color), (standardize: standardise) and so on. Depending upon the text data, the words from both of them can be present and one might want to convert all British english words to American words or vice versa.

In [ ]:
import requests

url ="https://raw.githubusercontent.com/hyperreality/American-British-English-Translator/master/data/american_spellings.json"
american_to_british_dict = requests.get(url).json()

url ="https://raw.githubusercontent.com/hyperreality/American-British-English-Translator/master/data/british_spellings.json"
british_to_american_dict = requests.get(url).json()

In [ ]:
# Based on https://stackoverflow.com/questions/42329766/python-nlp-british-english-vs-american-english
def britishize(text):
    text = [american_to_british_dict[word] if word in american_to_british_dict else word for word in text.split()]
    return " ".join(text)


def americanize(text):
    text = [british_to_american_dict[word] if word in british_to_american_dict else word for word in text.split()]
    return " ".join(text)

In [ ]:
text = "Discount analyse standardised colour"
americanize(text)

'Discount analyze standardized color'

In [ ]:
text = "'Discount analyze standardized color'"
britishize(text)

"'Discount analyse standardised color'"

In [ ]:
df["tweet_american"] = df["tweet_spellcheck_compound"].apply(americanize)
df["tweet_american"].head(10)

,tweet_american
0,deed reason earthquake may allah forgive a
1,airline horrible accident man died wing airline
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze muff built much hype around new acquisition doubt set cpl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rock fire update california hwy closed direction due lake county fire afire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


In [ ]:
df["tweet_british"] = df["tweet_spellcheck_compound"].apply(britishize)
df["tweet_british"].head(10)

,tweet_british
0,deed reason earthquake may allah forgive a
1,airline horrible accident man died wing airline
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze muff built much hype around new acquisition doubt set cpl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rock fire update california hwy closed direction due lake county fire afire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


## **Final Stopward Removal**

In [ ]:
df["tweet_final"] = df["tweet_spellcheck_compound"].apply(remove_stopwords)
df["tweet_final"].head(10)

,tweet_final
0,deed reason earthquake may allah forgive
1,airline horrible accident man died wing airline
2,ambulance twelve feared killed pakistani air ambulance helicopter crash
3,ablaze muff built much hype around new acquisition doubt set cpl ablaze season
4,resident asked shelter place notified officer evacuation shelter place order expected
5,people receive wildfire evacuation order california
6,got sent photo ruby alaska smoke wildfire pours school
7,rock fire update california hwy closed direction due lake county fire afire wildfire
8,flood disaster heavy rain cause flash flooding street manitou colorado spring area
9,top hill see fire wood


In [ ]:
df.to_csv("/content/drive/My Drive/NLP_DL/NLP/data/cleaned_sample.csv")

Do check the csv file generated after these steps in some external software like MS Excel or Google Docs to better understand the effects of each preprocessing step on the input text.

## Combined

In [ ]:
# Installs
# !pip install contractions
# !pip install symspellpy

In [ ]:
# Imports
import re
import string

import pandas as pd
import contractions
from unidecode import unidecode

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import pkg_resources
from symspellpy import SymSpell, Verbosity
import requests

In [ ]:
# Preparation
# For HTML removal with regex
HTML_ENTITIES = ["</\w+>", "<\w+>", "&nbsp;", "&lt;", "&gt;", "&amp;", "&quot;",
                 "&apos;", "&cent;", "&pound;", "&yen;", "&euro;", "&copy;", "&reg;",]

# For Stopwords
stop_words = set(stopwords.words('english'))

# For Lemmatization
lemmatizer = WordNetLemmatizer()

# For symspellpy
bigram_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_bigramdictionary_en_243_342.txt"
)
sym_spell.load_bigram_dictionary(bigram_path, term_index=0, count_index=2)

# For Britisize
url ="https://raw.githubusercontent.com/hyperreality/American-British-English-Translator/master/data/american_spellings.json"
american_to_british_dict = requests.get(url).json()

# For Americanize
url ="https://raw.githubusercontent.com/hyperreality/American-British-English-Translator/master/data/british_spellings.json"
british_to_american_dict = requests.get(url).json()

In [ ]:
def text_preprocessing(text):
    text = text.lower() # Lower
    for entity in HTML_ENTITIES: # Remove HTML
        text = re.sub(f"{entity}", " ", text)
    text = contractions.fix(text) # Expand contractions
    text = re.sub(re.compile(r'https?://(www\.)?(\w+)(\.\w+)(/\w*)?'), "", text) # Remove URL
    text = re.sub(re.compile(r"[\w\.-]+@[\w\.-]+\.\w+"), "", text) # Remove email
    text = re.sub(re.compile(r"@\w+"), "", text) # Remove Tweeter mentions
    text = unidecode(text) # Handle accented words
    text = text.encode("ascii", "ignore").decode() # Remove Unicode characters
    text = re.sub('[%s]' % re.escape(string.punctuation), " ",text)  # Remove Punctuations
    text = re.sub(re.compile("\w*\d+\w*"), "",text) # Remove digits and words with digits
#     text = re.sub(re.compile("[^A-Za-z0-9]"), " ", text)   # Remove Punctuations
#     text = re.sub(re.compile("[^A-Za-z]"), " ", text)   # Remove Punctuations and digits
    text = " ".join([word for word in str(text).split() if word not in stop_words]) # Remove stopwords
    text = re.sub(' +', ' ', text).strip()  # Remove extra space
    text = " ".join([lemmatizer.lemmatize(word) for word in text.split()]) # Lemmatization
    text = " ".join([sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2, include_unknown=True)[0].term for word in text.split()]) # Spelling correction
    text = " ".join([sym_spell.lookup_compound(word, max_edit_distance=2)[0].term for word in text.split()]) # Compount word correction
    text = " ".join([word for word in str(text).split() if word not in stop_words]) # Final stopwords removal
    text = " ".join([american_to_british_dict[word] if word in american_to_british_dict else word for word in text.split()]) # Britisize
    text = " ".join([british_to_american_dict[word] if word in british_to_american_dict else word for word in text.split()]) #Americanize

You can create a template with above code and fork it everytime you want to work on some NLP project.

## Sequence of Preprocessing Steps
Proper sequence of these operations need to be determined to achieve higher efficiency of data preprocessing

(will be added soon)

## Classification using Clean Dataset

I will add few examples of training multiple simple models and compare the results soon